# 첫 그래프 그리기

> 파이썬 9강 · 시각화

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [첫 그래프 그리기](https://mioon1402.github.io/timeseriesdata/python/p09-first-plot.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.**

In [ ]:
# 예시 데이터 내려받기
!wget -q -nc https://raw.githubusercontent.com/mioon1402/timeseriesdata/main/data/cafe_sales.csv

# 표를 글자로 찍을 때 한글 열이 어긋나지 않게 (한글을 두 칸으로 계산)
import pandas as pd
pd.set_option("display.unicode.east_asian_width", True)

# 그래프 한글 깨짐 방지
!pip install -q koreanize-matplotlib
import koreanize_matplotlib  # noqa: F401

print('준비 완료')

## 1. 그래프 하나 그려보기

**9-1. 첫 그래프**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("cafe_sales.csv", parse_dates=["date"])

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(df["date"], df["visitors"], linewidth=0.8)
ax.set_title("일별 방문객 추이")
ax.set_xlabel("날짜")
ax.set_ylabel("방문객 수(명)")
plt.show()

## 2. 한글이 깨질 때

**9-2. 폰트 확인해보기**

In [ ]:
print("현재 폰트:", plt.rcParams["font.family"])
print("음수 부호 처리:", plt.rcParams["axes.unicode_minus"])

fig, ax = plt.subplots(figsize=(7, 2.5))
ax.plot(df["date"], df["avg_temp"], linewidth=0.7, color="teal")
ax.set_title("기온 추이 — 한글과 음수 부호 확인")
ax.set_ylabel("평균기온(℃)")
ax.axhline(0, color="gray", linestyle="--", linewidth=1)
plt.show()

## 3. 네 가지 기본 그래프

**9-3. 선그래프 — 시간에 따른 변화**

In [ ]:
월별 = df.set_index("date")["sales"].resample("ME").mean() / 10000

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(월별.index, 월별.values, marker="o", markersize=4)
ax.set_title("월별 평균 매출")
ax.set_ylabel("매출(만원)")
plt.show()

**9-4. 막대그래프 — 범주 비교**

In [ ]:
순서 = ["월", "화", "수", "목", "금", "토", "일"]
요일평균 = df.groupby("weekday")["visitors"].mean().reindex(순서)

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(요일평균.index, 요일평균.values, color="#0d9488")
ax.set_title("요일별 평균 방문객")
ax.set_ylabel("방문객 수(명)")
plt.show()

**9-5. 히스토그램 — 분포 확인**

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(df["visitors"], bins=30, color="#d97706", edgecolor="white")
ax.set_title("방문객 수 분포")
ax.set_xlabel("방문객 수(명)")
ax.set_ylabel("일수")
plt.show()

**9-6. 산점도 — 두 변수의 관계**

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(df["avg_temp"], df["sales"] / 10000, s=10, alpha=0.4, color="#7c3aed")
ax.set_title("기온과 매출의 관계")
ax.set_xlabel("평균기온(℃)")
ax.set_ylabel("매출(만원)")
plt.show()

## 4. 어떤 그래프를 언제 쓰나

**9-7. 같은 데이터, 다른 인상**

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.4))

ax1.bar(요일평균.index, 요일평균.values, color="#dc2626")
ax1.set_ylim(140, 230)                    # 밑동을 잘랐다
ax1.set_title("✗ y축 140부터 — '토요일이 압도적'")

ax2.bar(요일평균.index, 요일평균.values, color="#0d9488")
ax2.set_ylim(0, 240)                      # 0부터
ax2.set_title("○ y축 0부터 — 실제 비율")

plt.tight_layout()
plt.show()

## 5. pandas 에서 바로 그리기

**9-8. .plot() 한 줄로**

In [ ]:
ax = 요일평균.plot(kind="bar", figsize=(7, 3), color="#2563eb",
                 title="요일별 평균 방문객", rot=0)
ax.set_ylabel("명")
plt.show()

**9-9. 여러 열을 한 번에**

In [ ]:
월별표 = (df.set_index("date")[["visitors", "sales"]]
            .resample("ME").mean())
월별표["sales"] = 월별표["sales"] / 10000

ax = 월별표.plot(figsize=(9, 3.2), secondary_y="sales",
                title="월별 방문객과 매출")
plt.show()

## 6. 그림 하나에 여러 그래프

**9-10. 2×2 대시보드**

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6.5))

axes[0, 0].plot(df["date"], df["visitors"], linewidth=0.6)
axes[0, 0].set_title("일별 방문객 추이")

axes[0, 1].bar(요일평균.index, 요일평균.values, color="#0d9488")
axes[0, 1].set_title("요일별 평균")

axes[1, 0].hist(df["visitors"], bins=30, color="#d97706", edgecolor="white")
axes[1, 0].set_title("방문객 분포")

axes[1, 1].scatter(df["avg_temp"], df["visitors"], s=8, alpha=0.35, color="#7c3aed")
axes[1, 1].set_title("기온 vs 방문객")

fig.suptitle("밀롱 커피 2년 요약", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**연습 · 직접 써보세요**

In [ ]:
# 문제 1. 월별 평균 기온을 선그래프로 그려보세요.


# 문제 2. 매출(sales)의 히스토그램을 그리고, 구간을 15개와 60개로
#        바꿔가며 모양이 어떻게 달라지는지 보세요.


# 문제 3. 방문객과 매출의 산점도를 그려보세요. 어떤 모양이 나오나요?

**모범 답안**

In [ ]:
# 문제 1
월기온 = df.set_index("date")["avg_temp"].resample("ME").mean()
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(월기온.index, 월기온.values, marker="o", color="teal")
ax.axhline(0, color="gray", ls="--", lw=1)
ax.set_title("월별 평균 기온"); ax.set_ylabel("℃")
plt.show()

# 문제 2
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3))
a.hist(df["sales"] / 10000, bins=15, color="#2563eb", edgecolor="white")
a.set_title("구간 15개")
b.hist(df["sales"] / 10000, bins=60, color="#2563eb", edgecolor="white")
b.set_title("구간 60개")
plt.tight_layout(); plt.show()

# 문제 3
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(df["visitors"], df["sales"] / 10000, s=8, alpha=0.4)
ax.set_xlabel("방문객(명)"); ax.set_ylabel("매출(만원)")
ax.set_title("방문객과 매출 — 거의 완벽한 직선")
plt.show()

---

전체 강의 목록 → [눈으로 보는 수학·통계](https://mioon1402.github.io/timeseriesdata/)